# Семинар 11 - динамические библиотеки

Наша программа, как правило, существует не сама по себе, а использует некоторые библиотеки. А значит, их надо как-то слинковать с нашим кодом.


## Статическая/динамическая линковка

При статической линковке библиотечный код вставляется в исполняемый файл (см. `make static.out`).

Можно проверить, что код функции `foo` действительно попал в `static.out` с помощью `objdump -d static.out | grep foo --after 9`.

При динамической линковке (см. `make dynamic.out`) код функции подгружается в момент выполнения специальной программой `ld`. Поскольку код библиотеки может быть загружен в произвольное место в адресном пространстве, он должен быть позиционно-независимым (`fPIC`). Можно убедиться, что `dynamic.out` не содержит `foo`.

При запуске нужно указать путь для поиска динамических библиотек `LD_LIBRARY_PATH=. ./dynamic.out`.

В качестве альтернативы можно зашить путь для поиска библиотек в исполняемый файл (см. цель `dynamic_rpath`).

Для того, чтобы посмотреть список динамических зависимостей файла, можно использовать `ldd`.

|Статическая линковка|Динамическая линковка|
|---|---|
|Происходит во время компиляции|Происходит во время исполнения|
|Код библиотеки содержится в исполняемом файле|Код библиотеки подгружается во время исполнения|
|Больше исполняемый файл|Меньше исполняемый файл|
|Медленнее загрузка|Быстрее загрузка|
|Тяжело поддерживать|Легко обновлять|

## Трамплины

Видно, что в `dynamic.out` содержится некая функция `foo@plt`. Она указывает на некоторый адрес.  Он называется `relocation` и указывает на то, что при вызове функции нужно её загрузить и адрес записать адрес функции. С помощью `readelf -r dynamic.out` можно увидеть релокацию для функции `foo`. При первом вызове из `plt` вызовется динамический линковщик, который загрузит функцию и проставит адрес в `got`.

## Загрузка библиотеки с помощью `dl`

`void *dlopen(const char *filename, int flags)` загружает файл с библиотекой и `handle`. Наиболее важные флаги:

* `RTLD_NOW` загружает все символы и только потом выходит из `dlopen`
* `RTLD_LAZY` подгружает символы по мере вызова

`void *dlsym(void *handle, const char *symbol)` возвращает арес символа

`int dlclose(void *handle)` уменьшает счётчик использований динамического объекта. Если он становится нулём, выгружает его из памяти.

`char *dlerror()` возвращает текст ошибки, связанной с динамической загрузкой библиотек.

Пример использования см. в `loader.c`.

## Загрузка библиотеки с помощью mmap

Как мы помним из прошлых занятий, можно делать маппинги произвольных файлов в память. В частности, можно загрузить код библиотеки с правами на исполнение и вызвать его. Пример чисто игрушечный (так в жизни делать не стоит) можно найти в `mmap_loader.c`.

# Контейнеры

Контейнер это набор изолированных процессов. Изоляция происходит на уровне файловой системы, сети, процессов и других ресурсов.

<img src="media/containers-vs-virtual-machines.jpg" alt="Отличия контейнеризации от виртуализации" width="600" style="background-color:white;"/>

В отличии от виртуальных машин, в которых поверх гипервизора запускается гостевая система целиком, включая ядро, контейнеры используют ядро хостовой ОС. Это требует меньше ресурсов, но взамен мы жертвуем возможностью эмулировать другие ОС (например Linux на Mac) и безопасностью.

## namespaces

Namespaces это компонент ядра Linux, который позволяет изолировать ресурсы. Категории ресурсов можно посмотреть в ``/proc/pid/ns``. Отделение части ресурсов процесса в новую группу происходит с помощью системного вызова ``unshare``.

### pid

<img src="media/pid_ns.png" alt="PID namespaces" width="600" style="background-color:white;"/>

Внутри контейнера видны только его процессы.

```
$  docker run -it --rm busybox
/ # ps aux
PID   USER     TIME  COMMAND
    1 root      0:00 sh
    7 root      0:00 ps aux
```

### mount

<img src="media/fs_ns.png" alt="Mount namespaces" width="600" style="background-color:white;"/>

```
/ # mount
overlay on / type overlay (rw,relatime,lowerdir=/var/lib/docker/overlay2/l/FT5QA66BOLCNWLHESUCX7GXQAF:/var/lib/docker/overlay2/l/QGKLLJVDLJEAQEPKFI5TNULEA7,upperdir=/var/lib/docker/overlay2/fd9a065e6c797fda3f02a9a7abca98cbce567d93698cb5d2fdd977c5f7c88b57/diff,workdir=/var/lib/docker/overlay2/fd9a065e6c797fda3f02a9a7abca98cbce567d93698cb5d2fdd977c5f7c88b57/work,nouserxattr)
...
```

Видно, что в корень контейнера смонтирована некая ``overlayfs``. Этот вид ФС использует ``Copy-on-write`` идиому. ``lower_dir`` это ``readonly`` слои ``image`` (команда ``RUN`` порождает новый слой, поэтому, по возможности, объединяйте команды в цепочки), которые можно переиспользовать для разных контейнеров. Когда вы меняете файл внутри контейнера, он записывается в ``upper_dir`` (в частности, если удалить файл из нижнего слоя, в верхнем появится файл-заглушка). ``Overlayfs`` на лету склеивает из слоев ``merged_dir``, именно ее вы и видите внутри контейнера.

Подробнее можно почитать в официальной документации: https://docs.docker.com/storage/storagedriver/overlayfs-driver/

### network

<img src="media/net_ns.png" alt="Network namespaces" width="600" style="background-color:white;"/>

По-умолчанию ``docker`` создает виртуальный свитч ``docker0``, который объединяет контейнеры и хост между собой.

```
$ ip addr
...
5: docker0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc noqueue state UP group default 
    link/ether 02:42:90:66:2a:b9 brd ff:ff:ff:ff:ff:ff
    inet 172.17.0.1/16 brd 172.17.255.255 scope global docker0
       valid_lft forever preferred_lft forever
...
``` 

Можно проверить, что контейнеры пингуются друг из друга. Если выключить интерфейс на хосте, внутри контейнера пропадет доступ в есть.

```
$ sudo ip link set docker0 down
```

Можно создавать отдельные свитчи для групп контейнеров: ``docker network create network_name`` (чтобы сеть появилась в контейнере, нужно ее передать через флаг ``--network network_name``). Контейнеры из разных сетей не видят друг друга. 

Еще один способ выключить сеть в контейнере это удалить маршрут со стороны хоста. 

```
$ sudo ip route del 172.19.0.0/16 dev br-68bd2dd5acc8
```

И, наконец, можно дропать все пакеты исходящие от контейнера с помощью ``iptables``.

```
$ sudo iptables -I FORWARD -s 172.19.0.2 -j DROP 
```

Неплохой гайд по разным вариантам настройки сети в контейнерах: https://k21academy.com/docker-kubernetes/docker-networking-different-types-of-networking-overview-for-beginners/


## cgroups

<img src="media/cgroups.jpeg" alt="Cgroups" width="600" style="background-color:white;"/>

Cgroups задают ограничения на ресурсы. Их иерархию можно посмотреть с помощью ``systemd-cgls``. Далее через ``sysfs`` можно смотреть информацию конкретной группе. Например: 

```
$ cat /sys/fs/cgroup/system.slice/containerd.service/pids.current 
36
```

## Toy containers

Пример минимальной реализации контейнеризации с помощью вышеописанных механизмов можно посмотреть по ссылке: https://github.com/carzil/containers/tree/master
